# MovieLens 32M Dataset


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import csr_matrix
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

In [ ]:
DATA_PATH = os.path.join(os.getcwd(), "..", "data", "ml-32m")
MODELS_PATH = os.path.join(os.getcwd(), "..", "models")

In [ ]:
links_df = pd.read_csv(os.path.join(DATA_PATH, "links.csv"))
movies_df = pd.read_csv(os.path.join(DATA_PATH, "movies.csv"))
ratings_df = pd.read_csv(os.path.join(DATA_PATH, "ratings.csv"))
tags_df = pd.read_csv(os.path.join(DATA_PATH, "tags.csv"))

print("About links CSV")
print(links_df.info())

print("\nAbout movies CSV")
print(movies_df.info())

print("\nAbout ratings CSV")
print(ratings_df.info())

print("\nAbout tags CSV")
print(tags_df.info())

In [ ]:
rating_counts = ratings_df["movieId"].value_counts()
print(rating_counts.quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]))

In [ ]:
# Filtrar películas poco populares para optimizar memoria
min_ratings = 3
valid_movie_ids = rating_counts[rating_counts >= min_ratings].index
ratings_filtered = ratings_df[ratings_df["movieId"].isin(valid_movie_ids)].copy()
total_movies_used_pr = (len(valid_movie_ids) / movies_df["movieId"].count()) * 100

print(f"Total movies used: {total_movies_used_pr}% of the catalog.")

In [ ]:
# 3. Mapear IDs originales a índices continuos
user_mapping = {id_: i for i, id_ in enumerate(ratings_filtered["userId"].unique())}
movie_mapping = {id_: i for i, id_ in enumerate(ratings_filtered["movieId"].unique())}

ratings_filtered["user_idx"] = ratings_filtered["userId"].map(user_mapping)
ratings_filtered["movie_idx"] = ratings_filtered["movieId"].map(movie_mapping)

In [ ]:
# 4. Crear la matriz dispersa Usuario-Película
num_users = len(user_mapping)
num_movies = len(movie_mapping)

R_sparse = csr_matrix(
    (
        ratings_filtered["rating"],
        (ratings_filtered["user_idx"], ratings_filtered["movie_idx"]),
    ),
    shape=(num_users, num_movies),
)

In [ ]:
ratings_per_user: np.ndarray = R_sparse.getnnz(axis=1)
print("Mínimo de ratings por usuario:", ratings_per_user.min())
print("Máximo de ratings por usuario:", ratings_per_user.max())
print("Media de ratings por usuario:", ratings_per_user.mean())

In [ ]:
normalizer = Normalizer(norm="l2")
R_scaled = normalizer.fit_transform(R_sparse)

In [ ]:
# 5. Entrenar el modelo NMF
# Se recomienda solver='cd' e init='nndsvda' para matrices dispersas grandes
nmf = NMF(
    n_components=50,
    solver="cd",
    init="nndsvda",
    max_iter=500,
    random_state=42,
)

W = nmf.fit_transform(R_sparse)
H = nmf.components_  # Forma: (componentes, películas)

In [ ]:
# Guardar el modelo entrenado para uso futuro
joblib.dump(nmf, os.path.join(MODELS_PATH, "nmf_model_n50_iter500.pkl"))
joblib.dump(W, os.path.join(MODELS_PATH, "nmf_W_n50_iter500.pkl"))
joblib.dump(H, os.path.join(MODELS_PATH, "nmf_H_n50_iter500.pkl"))

In [ ]:
# # Carga del modelo entrenado (si es necesario)
# nmf = joblib.load(os.path.join(MODELS_PATH, "nmf_model_n50_iter500.pkl"))
# W = joblib.load(os.path.join(MODELS_PATH, "nmf_W_n50_iter500.pkl"))
# H = joblib.load(os.path.join(MODELS_PATH, "nmf_H_n50_iter500.pkl"))

In [ ]:
# 6. Matriz de películas en el espacio latente (Películas x Componentes)
movie_features = H.T

# 7. Calcular la matriz de similitud de coseno
similarity_matrix = cosine_similarity(movie_features)

In [ ]:
# Mapeos inversos para búsqueda por título
movie_to_idx = dict(zip(movies_df["title"], movies_df["movieId"].map(movie_mapping)))
idx_to_movie = {
    idx: movies_df.loc[movies_df["movieId"] == original_id, "title"].values[0]
    for original_id, idx in movie_mapping.items()
}

In [ ]:
def get_recommendations(title, top_n=10):
    if title not in movie_to_idx or pd.isna(movie_to_idx[title]):
        return f"La película '{title}' no está disponible o no superó el umbral de valoraciones."

    movie_idx = int(movie_to_idx[title])

    # Obtener puntuaciones de similitud para la película
    sim_scores = list(enumerate(similarity_matrix[movie_idx]))

    # Ordenar de mayor a menor similitud (excluyendo la propia película)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1 : top_n + 1]

    # Recuperar títulos e índices
    recommendations = [(idx_to_movie[i], score) for i, score in sim_scores]
    return pd.DataFrame(recommendations, columns=["Película", "Similitud"])

In [ ]:
# Ejemplo de uso
print(get_recommendations("Toy Story (1995)", top_n=5))